# 01 — Profiling del dataset
**Spec:** docs/specs/01_profiling_y_arquitectura.md · **Requisitos:** R-01, R-09, R-15, R-40 · **Escribe:** nada

Evidencia de cada decisión de diseño, recalculada en Spark sobre el CSV completo. Cada celda cierra con el ADR que justifica.
Los outputs no muestran `id_cliente` ni `id_transaccion`: este notebook se publica.

In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog", "workspace")

import json
import os
import sys
import uuid

sys.path.insert(0, os.path.abspath("../src"))
from iol_pipeline import config, io, profiling, segmentation

RUN_ID = dbutils.widgets.get("pipeline_run_id") or str(uuid.uuid4())
CATALOG = dbutils.widgets.get("catalog")
print("pipeline_run_id:", RUN_ID)

trades_dir = io.landing_path("trades", catalog=CATALOG)
raw = profiling.read_trades_csv(spark, trades_dir)
trades = profiling.prepare(raw)

pipeline_run_id: 53edb4db-e774-4be9-839f-14d330e6ed16


## 1. Conteos, nulos, duplicados y rango temporal

In [0]:
summary = profiling.general_summary(raw, trades)
for key, value in summary.items():
    print(f"{key:24} {value}")

rows                     100000
clients                  57655
symbols                  1445
distinct_trade_ids       100000
first_local              2026-01-02 07:33:18
last_local               2026-03-13 16:59:48
unparseable_ts           0
duplicate_trade_ids      0
nulls_by_column          {'fecha': 0, 'tipoTran': 0, 'id_cliente': 0, 'descripcion_titulo': 0, 'moneda': 0, 'simbolo_titulo': 0, 'cantidad': 0, 'precio': 0, 'id_transaccion': 0, 'origen': 0}


→ **Decisión:** marzo es un período parcial, cortado en `DATA_CUTOFF`. Las comparaciones entre meses se normalizan por día hábil (celda 9).

## 2. Día de semana y días hábiles sin operaciones

In [0]:
display(profiling.weekday_distribution(trades))
print("Días lun-vie sin operaciones:", profiling.weekdays_without_trades(trades))
print("Operaciones en feriados del calendario:", profiling.trades_on_market_holidays(trades))

weekday,name,ops_utc,ops_local
1,lun,9155,11506
2,mar,20142,19749
3,mie,19099,19178
4,jue,19487,19438
5,vie,19619,19874
6,sab,12498,10255
7,dom,0,0


Días lun-vie sin operaciones: [{'date': '2026-02-16', 'market_holiday': 'Carnaval'}, {'date': '2026-02-17', 'market_holiday': 'Carnaval'}]
Operaciones en feriados del calendario: 0


→ **Decisión: ADR-003.** Si el dataset respeta el calendario, un sábado no se explica como operación fuera de mercado cualquiera.

## 3. Hora UTC por día de semana

In [0]:
display(profiling.hourly_by_weekday(trades))
display(profiling.hour_bounds_by_weekday(trades))
print(f"Proporción dentro de la rueda {config.SESSION_HOURS_LOCAL} local: {profiling.share_in_session(trades):.1%}")

trade_hour_utc,Mon,Tue,Wed,Thu,Fri,Sat,Sun
0,0,783,633,666,624,771,0
1,0,769,640,696,664,742,0
2,0,799,685,675,700,730,0
3,0,762,682,734,690,744,0
4,0,768,671,700,678,778,0
5,0,746,629,702,655,722,0
6,0,809,658,739,631,729,0
7,0,851,615,714,638,755,0
8,0,765,661,717,668,788,0
9,0,785,702,789,671,744,0


weekday,first_hour_utc,last_hour_utc,ops
Wed,0,23,19099
Thu,0,23,19487
Mon,10,23,9155
Sat,0,19,12498
Tue,0,23,20142
Fri,0,23,19619


Proporción dentro de la rueda ('11:00', '17:00') local: 29.6%


→ **Decisión: ADR-003.** Se compara dónde arranca el lunes y dónde corta el sábado. La hora no se usa para análisis intradiario y `DQ013` queda como métrica INFO.

## 4. Canales

In [0]:
display(profiling.channel_counts(trades))
print("Clientes multicanal:", profiling.multichannel_clients(trades))

origen,ops
App Mobile,66438
Sitio Web Desktop,28952
Sitio Web Responsive,2651
API,1688
IOLnet,271


Clientes multicanal: 1741


→ **Decisión:** el canal va en el hecho, y `preferred_channel` en el SCD2 (ADR-007).

## 5. Actividad por cliente

In [0]:
activity = profiling.client_activity(trades)
for key, value in activity.items():
    print(f"{key:18} {value}")

percentiles        {0.5: 1.0, 0.75: 2.0, 0.9: 3.0, 0.95: 4.0, 0.99: 10.0}
max_ops            676
clients            57655
one_trade          40841
one_trade_share    0.7083687451218454


→ **Decisión: ADR-007.** Los cortes de `activity_segment` se ubican en la cola de la distribución.

## 6. Instrumentos, sufijos y descripciones
La clasificación por tipo es preliminar; la definitiva sale del spec 04.

In [0]:
display(profiling.asset_type_by_currency(trades))
for key, value in profiling.instrument_summary(trades).items():
    print(f"{key:32} {value}")
print(f"Top {config.PROFILE_TOP_SYMBOLS} símbolos: {profiling.top_symbols_share(trades, config.PROFILE_TOP_SYMBOLS):.1%} de las operaciones")

asset_type_rule,ARS,USD
BONO_SOBERANO,12446,5764
BONO_SUBSOBERANO,229,34
CEDEAR,47610,6442
LETRA_LECAP_BONCAP,3912,50
NEEDS_REVIEW,18012,1229
ON,1292,1061
OPCION,1824,0
WARRANT,95,0


usd_symbols                      471
usd_with_ars_base                279
ars_ending_in_d_or_c             ['AMD', 'BA.C', 'BA37D', 'BB37D', 'BBD', 'BC37D', 'C', 'CARC', 'DD', 'ERIC', 'GILD', 'GLD', 'HD', 'HMC', 'HOOD', 'HSBC', 'INTC', 'JD', 'KGC', 'LAC', 'LND', 'MCD', 'PDD', 'SA24D', 'SID', 'VOD', 'WFC', 'XLC', 'YPFD']
symbols_with_many_descriptions   ['AZN', 'IRS2W', 'X15Y6']
Top 20 símbolos: 38.9% de las operaciones


→ **Decisión: ADR-004.** El sufijo D/C no alcanza para identificar variantes: el mapeo es una tabla explícita.

## 7. Precios por variante de liquidación y escala
`implied_fx` = mediana del símbolo base / mediana de la variante. No se comparan precios entre monedas para sumarlos: solo se muestra la relación.

In [0]:
display(profiling.settlement_price_medians(trades))

base,simbolo_titulo,moneda,median_price,ops,implied_fx
AAPL,AAPL,ARS,19625.8112,1110,null
AAPL,AAPLC,USD,13.1884,1,1488.111613235874
AAPL,AAPLD,USD,13.4356,152,1460.732025365447
AL30,AL30,ARS,876.3192,8859,null
AL30,AL30C,USD,0.59035,264,1484.4061997120352
AL30,AL30D,USD,0.6073,4589,1442.975794500247
GD30,GD30,ARS,911.583,95,null
GD30,GD30D,USD,0.61825,48,1474.4569348968864


→ **Decisión: ADR-005.** Si el `implied_fx` de los bonos es coherente con el de los CEDEARs, la escala del dataset es por 1 VN. Cada variante es un instrumento distinto (ADR-004).

## 8. Extremos y posibles errores de escala
El top por importe va por moneda: ARS y USD no se comparan.

In [0]:
display(profiling.top_by_quantity(trades, config.PROFILE_TOP_N))
display(profiling.top_by_amount(trades, config.PROFILE_TOP_N))
display(profiling.treasury_capitalizable_prices(trades))
anomalies = profiling.scale_anomalies(trades)
print("Filas con precio fuera del rango de DQ021:", anomalies.count())
display(anomalies)

simbolo_titulo,origen,asset_type_rule,moneda,cantidad,precio,amount
T30A7,IOLnet,BONO_SOBERANO,ARS,1.503210011E9,0.0101,1.51824211111E7
T31Y7,IOLnet,BONO_SOBERANO,ARS,1.427140785E9,0.01,1.427140785E7
TTS26,IOLnet,BONO_SOBERANO,ARS,1.067195974E9,1.3802,1.4729438833148E9
T30J6,IOLnet,LETRA_LECAP_BONCAP,ARS,9.76270797E8,1.255,1.225219850235E9
T13F6,IOLnet,LETRA_LECAP_BONCAP,ARS,6.61750591E8,1.4238,9.422004914657999E8
T13F6,IOLnet,LETRA_LECAP_BONCAP,ARS,4.96675425E8,1.4189,7.047327605325E8
T13F6,IOLnet,LETRA_LECAP_BONCAP,ARS,3.31012551E8,0.01,3310125.5100000002
T13F6,IOLnet,LETRA_LECAP_BONCAP,ARS,2.8989975E8,1.4029,4.0670035927500004E8
S17A6,IOLnet,LETRA_LECAP_BONCAP,ARS,2.13473214E8,1.0341,2.207526505974E8
S17A6,IOLnet,LETRA_LECAP_BONCAP,ARS,1.71938257E8,1.0261,1.764258455077E8


rank,simbolo_titulo,origen,asset_type_rule,moneda,cantidad,precio,amount
1,TTS26,IOLnet,BONO_SOBERANO,ARS,1.067195974E9,1.3802,1.4729438833148E9
2,T30J6,IOLnet,LETRA_LECAP_BONCAP,ARS,9.76270797E8,1.255,1.225219850235E9
3,T13F6,IOLnet,LETRA_LECAP_BONCAP,ARS,6.61750591E8,1.4238,9.422004914657999E8
4,T13F6,IOLnet,LETRA_LECAP_BONCAP,ARS,4.96675425E8,1.4189,7.047327605325E8
5,T13F6,IOLnet,LETRA_LECAP_BONCAP,ARS,2.8989975E8,1.4029,4.0670035927500004E8
6,TZXM6,IOLnet,BONO_SOBERANO,ARS,1.43927228E8,2.0675,2.9756954389E8
7,SPY,Sitio Web Desktop,CEDEAR,ARS,4267.0,52534.6681,2.241654287827E8
8,S17A6,IOLnet,LETRA_LECAP_BONCAP,ARS,2.13473214E8,1.0341,2.207526505974E8
9,TRAN,Sitio Web Desktop,NEEDS_REVIEW,ARS,50977.0,3757.5387,1.9154805030990002E8
10,S17A6,IOLnet,LETRA_LECAP_BONCAP,ARS,1.71938257E8,1.0261,1.764258455077E8


simbolo_titulo,ops,min_price,median_price,max_price,min_over_median
T13F6,143,0.01,1.412,1.4528,0.007082152974504249
T30A7,127,0.0101,1.0814,1.1379,0.00933974477529129
T31Y7,35,0.01,1.0217,1.0757,0.00978760888714887
T30J7,16,1.0136,1.0697999999999999,1.1105,0.9474668162273324
S17A6,694,0.9962,1.04095,1.0753,0.9570104231711417
M31G6,190,1.0588,1.10435,1.1582,0.9587540182007516
S30A6,291,1.141,1.188,1.2342,0.9604377104377105
T30J6,112,1.2367,1.2772000000000001,1.3337,0.9682900093955527
S31L6,96,1.0002,1.0283,1.0616,0.9726733443547603
S31G6,126,1.0366,1.063,1.1134,0.9751646284101599


Filas con precio fuera del rango de DQ021: 13


simbolo_titulo,origen,asset_type_rule,moneda,cantidad,precio,amount,symbol_median,price_ratio
GFGC75054F,API,OPCION,ARS,422.0,1.0722,452.46840000000003,95.7231,0.011201058051818213
AL30,IOLnet,BONO_SOBERANO,ARS,426.0,0.01,4.26,876.3192,1.141136699960471E-5
DNC7D,IOLnet,ON,USD,337417.0,0.01,3374.17,1.0712,0.009335324869305453
METR,IOLnet,NEEDS_REVIEW,ARS,40351.0,0.9997,40338.894700000004,2252.8243,4.437540912533658E-4
SATL,IOLnet,CEDEAR,ARS,930.0,1.0046,934.2779999999999,5355.4863,1.875833386036297E-4
T13F6,IOLnet,LETRA_LECAP_BONCAP,ARS,3.31012551E8,0.01,3310125.5100000002,1.412,0.007082152974504249
T30A7,IOLnet,BONO_SOBERANO,ARS,1.503210011E9,0.0101,1.51824211111E7,1.0814,0.00933974477529129
T31Y7,IOLnet,BONO_SOBERANO,ARS,1.427140785E9,0.01,1.427140785E7,1.0217,0.00978760888714887
TGNO4,IOLnet,NEEDS_REVIEW,ARS,38077.0,0.9999,38073.1923,4335.50765,2.306304314789988E-4
GFGC75054F,Sitio Web Desktop,OPCION,ARS,1733.0,0.931,1613.423,95.7231,0.009725970011418351


→ **Decisión: DQ021 y ADR-005.** Se marcan y no se corrigen: no se sabe si el error está en el precio, en la cantidad o en ambos.

## 9. Operaciones por día hábil observado

In [0]:
display(profiling.monthly_activity(trades))

month,ops,business_days_observed,ops_per_business_day
2026-01,47562,21,2264.8571428571427
2026-02,35171,18,1953.9444444444443
2026-03,17267,10,1726.7


→ **Decisión:** las comparaciones entre meses (q3, q4) usan operaciones por día hábil observado, porque marzo está cortado.

## 10. Viabilidad del SCD2 (ADR-007)
Mide si hay cambios reales de atributos para demostrar el SCD2 con datos del dataset. **No hay valores esperados.**

In [0]:
print(f"""Definición aplicada ({config.SEGMENT_RULE_VERSION}):
- Versión inicial en la primera operación del cliente, con atributos de esa sola operación.
- Cortes acumulados: {[c.isoformat() for c in config.SCD2_SNAPSHOTS]}, por fecha local efectiva.
- activity_segment con los cortes congelados: {config.ACTIVITY_SEGMENTS}.
- preferred_channel = channel_code con más operaciones hasta el corte; desempate: operación más reciente, después alfabético.
- Cuentan las operaciones con cliente y fecha efectiva válidos; un duplicado exacto una vez; sin IDs en conflicto.
- Una versión nueva = cambia activity_segment o preferred_channel. dominant_asset_type y currency_profile no entran acá
  (dependen del spec 04): sumarlos solo puede agregar versiones, así que (a) es un piso.""")

usable = segmentation.usable_trades(trades)
usable_rows = usable.count()
print(f"Filas usadas: {usable_rows:,} de {summary['rows']:,}. Excluidas según ADR-007: {summary['rows'] - usable_rows:,}")
points = segmentation.client_points(usable)
distribution = segmentation.versions_distribution(points)

Definición aplicada (seg_v1):
- Versión inicial en la primera operación del cliente, con atributos de esa sola operación.
- Cortes acumulados: ['2026-01-31', '2026-02-28', '2026-03-13'], por fecha local efectiva.
- activity_segment con los cortes congelados: [(10, 'MUY_ACTIVO'), (4, 'ACTIVO'), (2, 'RECURRENTE'), (1, 'OCASIONAL')].
- preferred_channel = channel_code con más operaciones hasta el corte; desempate: operación más reciente, después alfabético.
- Cuentan las operaciones con cliente y fecha efectiva válidos; un duplicado exacto una vez; sin IDs en conflicto.
- Una versión nueva = cambia activity_segment o preferred_channel. dominant_asset_type y currency_profile no entran acá
  (dependen del spec 04): sumarlos solo puede agregar versiones, así que (a) es un piso.
Filas usadas: 100,000 de 100,000. Excluidas según ADR-007: 0


### (a) Clientes por número de versiones

In [0]:
display(distribution)
print("Clientes con 2 o más versiones:", sum(r.clients for r in distribution.collect() if r.versions >= 2))

versions,clients
1,40841
2,14435
3,2312
4,67


Clientes con 2 o más versiones: 16814


### (b) Cambios entre puntos consecutivos
Entre clientes presentes en ambos puntos. Las filas `inicial → 2026-02-28` e `inicial → 2026-03-13` son los clientes nacidos después del corte anterior.

In [0]:
display(segmentation.transitions(points))

from_point,point,clients,channel_changes,segment_changes
inicial,2026-01-31,31798,458,7480
2026-01-31,2026-02-28,31798,401,5025
inicial,2026-02-28,17926,147,2626
2026-02-28,2026-03-13,49724,277,3306
inicial,2026-03-13,7931,34,729


### (c) Matriz de transición de `activity_segment`

In [0]:
display(segmentation.segment_transition_matrix(points))

from_point,point,from_segment,activity_segment,clients
inicial,2026-01-31,OCASIONAL,ACTIVO,1276
inicial,2026-01-31,OCASIONAL,MUY_ACTIVO,184
inicial,2026-01-31,OCASIONAL,OCASIONAL,24318
inicial,2026-01-31,OCASIONAL,RECURRENTE,6020
2026-01-31,2026-02-28,ACTIVO,ACTIVO,1005
2026-01-31,2026-02-28,ACTIVO,MUY_ACTIVO,271
2026-01-31,2026-02-28,MUY_ACTIVO,MUY_ACTIVO,184
2026-01-31,2026-02-28,OCASIONAL,ACTIVO,307
2026-01-31,2026-02-28,OCASIONAL,MUY_ACTIVO,7
2026-01-31,2026-02-28,OCASIONAL,OCASIONAL,20671


→ **Decisión: ADR-007.** Si (b) da 0 o (a) da muy poco, se replantea ADR-007 antes del spec 06.

In [0]:
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "pipeline_run_id": RUN_ID, "rows_out": 0, "rows_read": summary["rows"]}))